In [0]:
# Step 122: Silver pipeline runner with failure logging
import uuid
from datetime import datetime, timezone

from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType, LongType
)

log_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("load_type", StringType(), True),
    StructField("processing_date", StringType(), True),
    StructField("source_folder", StringType(), True),
    StructField("start_timestamp", TimestampType(), True),
    StructField("end_timestamp", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("rows_read", LongType(), True),
    StructField("rows_inserted", LongType(), True),
    StructField("rows_updated", LongType(), True),
    StructField("rows_quarantined", LongType(), True),
    StructField("error_message", StringType(), True),
])

# Runner parameters (same values as the Bronze runner)
dbutils.widgets.dropdown("load_type", "incremental", ["full", "incremental"])
dbutils.widgets.text("processing_date", "")
dbutils.widgets.text("source_folder", "")

load_type = dbutils.widgets.get("load_type")
processing_date = dbutils.widgets.get("processing_date").strip()
source_folder = dbutils.widgets.get("source_folder").strip()

runner_run_id = str(uuid.uuid4())
runner_start_time = datetime.now(timezone.utc).replace(tzinfo=None)
log_table = "workspace.default.pipeline_execution_logs"

try:
    print("Starting Silver pipeline...")
    result = dbutils.notebook.run(
        "./02_silver_pipeline_final",
        0,
        {"load_type": load_type, "processing_date": processing_date, "source_folder": source_folder},
    )
    print("Silver pipeline completed successfully!")
    print("Notebook result:", result)

except Exception as e:
    failure_df = spark.createDataFrame(
        [(
            runner_run_id, "silver", "workspace.default.silver_market_observation",
            load_type, processing_date or None, source_folder or None,
            runner_start_time, datetime.now(timezone.utc).replace(tzinfo=None),
            "FAILED", 0, 0, 0, 0, str(e)[:4000],
        )],
        schema=log_schema,
    )
    (
        DeltaTable.forName(spark, log_table).alias("target")
        .merge(failure_df.alias("source"), "target.run_id = source.run_id")
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Silver pipeline FAILED. Failure recorded in execution logs.")
    raise

In [0]:
# Step 123: Latest Silver execution logs
display(
    spark.table("workspace.default.pipeline_execution_logs")
    .filter(F.col("layer") == "silver")
    .orderBy(F.col("end_timestamp").desc())
    .select("run_id", "load_type", "processing_date", "status", "rows_read",
            "rows_inserted", "rows_updated", "rows_quarantined", "start_timestamp", "end_timestamp")
    .limit(5)
)

In [0]:
# Step 124: Consistency checks (no hard-coded row counts, so it also passes after incremental loads)
bronze_final = spark.table("workspace.default.bronze_coin_market_raw")
silver_final = spark.table("workspace.default.silver_market_observation")
quarantine_final = spark.table("workspace.default.silver_quarantine")

bronze_count = bronze_final.count()
bronze_distinct_keys = bronze_final.select("coin_id", "observation_timestamp").distinct().count()
silver_count = silver_final.count()
quarantine_count = quarantine_final.count()

duplicate_count = (
    silver_final.groupBy("coin_id", "observation_timestamp").count()
    .filter(F.col("count") > 1).count()
)

print("Bronze rows:", bronze_count, "| distinct keys:", bronze_distinct_keys)
print("Silver rows:", silver_count)
print("Quarantine rows:", quarantine_count)
print("Duplicate Silver business keys:", duplicate_count)

if duplicate_count == 0 and silver_count + quarantine_count == bronze_distinct_keys:
    print("PASS: Silver + quarantine account for every distinct Bronze key, with no duplicates.")
else:
    print("CHECK REQUIRED: counts do not reconcile (a layer may be missing a batch).")